# Haru-4B — full-duplex speech-to-speech on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Haru-4B-full-duplex-speech.ipynb)

[`higurashi-ai/Haru-4B`](https://huggingface.co/higurashi-ai/Haru-4B) is higurashi's **pretrained BASE** full-duplex spoken dialogue model. [Apache-2.0](https://huggingface.co/higurashi-ai/Haru-4B), public (not gated). The Hub lists **5,270M** stored parameters including the audio codec (`model.safetensors` **10.5 GB**, dtype **bfloat16**). The language-model backbone is **Qwen3.5-4B** (32 layers, hidden size 2560). Audio is the causal **VibeVoice** acoustic tokenizer at **24 kHz**, **7.5 Hz** frames of **133 ms** (**3200 samples** per latent). A 4-layer MLP **MeanFlow** head samples each 64-dimensional audio latent in **one step** (guidance scale 1–2, default 1.5). Technical blog dated **13 November 2026**.

**Why this model.** It listens and speaks at the same time: every 133 ms it takes one frame of the user's audio and produces one frame of its own audio and text. User audio, model audio, and model text are summed into a single language-model position, so listening and speaking never wait for each other. That is the card's `DuplexSession` API, not an ASR → LLM → TTS pipeline. A **character card** with a fixed layout sets the persona; a **short reference clip** sets the voice. Haru ships **without** voice clips. Nine languages on the card: English, Spanish, Portuguese, German, French, Chinese, Japanese, Russian, and Korean.

This notebook is the **offline** Colab path. A free Colab runtime cannot open a live microphone duplex loop. We feed a prerecorded utterance in 3200-sample chunks, pad with silence so the model can reply, then save/play the output audio and print the text stream.

**This is a research base model, not an instruction follower.** The card: it can hold a natural conversation, but it is not a reliable source of facts and does not follow instructions the way an assistant does.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Haru-4B-full-duplex-speech.md) for when this notebook is a good fit and when it is not.

**Licence:** [Apache-2.0](https://huggingface.co/higurashi-ai/Haru-4B) on the weights and the [Haru](https://github.com/higurashi-ai/Haru) code.

**Consent:** only use a voice you have the right to use. The reference clips below are public-domain LibriVox audiobook readings from [`hf-internal-testing/librispeech_asr_dummy`](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy) (same family as OpenSLR LibriSpeech, CC BY 4.0). They are **not** a product voice.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. The card says an NVIDIA GPU needs about **12 GB**. A free T4 has ~15 GB and **no native bfloat16**. 5.27B parameters in float16 are ~**9.8 GB**; the Hub shard is ~10.5 GB of stored bfloat16. The card helper `HaruModel.from_pretrained(..., device="cuda", dtype=torch.float16)` **does not fit a T4**: it `load_file`s the whole bf16 shard onto the GPU (~10.5 GB) and then `.to(dtype=fp16)` while that copy is still alive, which is what OOM'd at ~14.4 GB allocated. This notebook builds the module on the **meta** device in float16 and copies each tensor to CUDA already as float16, so only one copy sits in VRAM (~10–11.5 GB after load, plus CUDA context). Non-persistent RoPE / MeanFlow buffers are not in the shard; they are recomputed on CUDA so a later `.to()` does not hit leftover meta tensors. The checkpoint is bf16-trained and Qwen3.5 gated delta-net is fp16-unstable, so after load this notebook **keeps fp16 weights** and runs linear-attention, norms, logits, softmax, and the MeanFlow head in **fp32** (T4 Tensor Cores are fp16; emulated bf16 is slower at the same 2-byte storage). A sanity cell before Demo 1 asserts logits are finite. First run downloads ~10.5 GB. Codec backend is PyTorch (Triton needs bfloat16 on CUDA). **After the install cell, restart the session** before loading.

**Not in this notebook:** the web demo / live mic, the MLX macOS app, vLLM-Omni serving, or Triton codec kernels.

**Links:** [model card](https://huggingface.co/higurashi-ai/Haru-4B) · [GitHub](https://github.com/higurashi-ai/Haru) · [technical blog](https://blog.higurashi.ai/melte/haru4b/) · [announcement](https://x.com/HigurashiAI/status/2121052314773717386) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Haru-4B-full-duplex-speech.md)


## 0. Early GPU check

Print `torch.cuda.is_available()`, the GPU name, and free VRAM before installing anything. Need a CUDA GPU (T4 is the free-Colab target). A restart after the install cell drops this state; section 2 prints it again and sets `DEVICE` / `DTYPE`.


In [ ]:
import shutil

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; this notebook will load float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install Haru (keep Colab's torch)

The [model card](https://huggingface.co/higurashi-ai/Haru-4B#usage) install is:

```bash
git clone https://github.com/higurashi-ai/Haru && cd Haru
pip install -e ./haru
```

This cell clones that repo, pins commit `7fe6b7665f8bf035998f401f8cb779fbe53f2751` (`haru-speech` 0.1.1), and installs the PyTorch package. `pyproject.toml` wants `transformers>=5.16` (Qwen3.5 text backbone). Current Colab already has a CUDA `torch` (`torch>=2.6` is enough) — this cell does **not** replace it.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Skip the `triton`, `webrtc`, and `share` extras; this notebook uses the default PyTorch codec backend.

**Do not import `haru` in this cell.** Colab has already imported its preinstalled `transformers`; the upgrade is invisible until the process restarts. The install cell restarts the session for you. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata
import os
import signal
import subprocess
import sys
from pathlib import Path

HARU_COMMIT = "7fe6b7665f8bf035998f401f8cb779fbe53f2751"
HARU_DIR = Path("/content/Haru") if Path("/content").is_dir() else Path("Haru").resolve()
MARKER = Path("/content/.haru_nb_install_ok") if Path("/content").is_dir() else Path(".haru_nb_install_ok")
already_restarted = MARKER.exists() and MARKER.read_text().strip() == HARU_COMMIT

print("python", sys.version.split()[0])

try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

if not (HARU_DIR / ".git").is_dir():
    subprocess.check_call(["git", "clone", "https://github.com/higurashi-ai/Haru.git", str(HARU_DIR)])
subprocess.check_call(["git", "-C", str(HARU_DIR), "fetch", "--depth", "1", "origin", HARU_COMMIT])
subprocess.check_call(["git", "-C", str(HARU_DIR), "checkout", "--detach", HARU_COMMIT])
print("Haru commit", subprocess.check_output(["git", "-C", str(HARU_DIR), "rev-parse", "HEAD"], text=True).strip())

# Card: pip install -e ./haru. Keep Colab's CUDA torch. Use %pip, not python -I.
# Do not `import haru` here — that needs the new transformers, which Colab will
# not see until this process restarts.
haru_pkg = str(HARU_DIR / "haru")
get_ipython().run_line_magic("pip", f"install -q -e {haru_pkg} --no-deps")
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers>=5.16" "tokenizers>=0.21" "safetensors>=0.4" '
    '"huggingface-hub>=0.26" "numpy>=1.26" "scipy>=1.13" "soundfile>=0.13" datasets',
)

print("HARU_DIR", HARU_DIR)
print("haru-speech", importlib.metadata.version("haru-speech"))
print("transformers (installed)", importlib.metadata.version("transformers"))
MARKER.write_text(HARU_COMMIT)

if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so the new transformers is what gets imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> **Restart the session now** (*Runtime → Restart session*), then continue from the **next** cell. Do not rerun the install unless it failed before printing package versions. The git clone and pip install survive on disk. The install cell also restarts the process for you (`os.kill`); a Colab "session crashed" or "Operation cancelled by user" right after the versions print is that restart, not a failed install. After reconnect, skip the install cell.


## 2. Runtime check and T4 dtype

Need a CUDA GPU. Print `torch.cuda.is_available()`, the device name, and free VRAM again (a restart dropped the first cell's state). Pick **float16** on a T4 and **bfloat16** only when the GPU has native BF16 (sm80+). On a T4 we keep fp16 weights (Tensor Cores) and upcast the overflow-prone ops after load — we do not use emulated bf16 as the default. Import `haru` here, after the restart, so it sees `transformers>=5.16`. The Triton codec backend is **not** used (it needs bfloat16 on CUDA).


In [ ]:
import os
import shutil
from pathlib import Path

import torch
import transformers
from packaging.version import Version

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

tf_ver = Version(transformers.__version__)
if tf_ver < Version("5.16"):
    raise SystemExit(
        f"transformers {transformers.__version__} is loaded; Haru needs >=5.16 "
        "(Qwen3.5 text backbone). Run the install cell, then Runtime → Restart session."
    )

import haru

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("haru", haru.__version__)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime → Change runtime type → T4 GPU, then rerun from the GPU-check cell.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")

# Checkpoint dtype is bfloat16. T4 has no native BF16 — use float16 there.
NATIVE_BF16 = major >= 8
DEVICE = "cuda"
DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
MODEL_ID = "higurashi-ai/Haru-4B"
SAMPLE_RATE = 24_000
FRAME_SAMPLES = 3200  # 133.3 ms at 24 kHz; card / config.json
FRAME_MS = 1000.0 * FRAME_SAMPLES / SAMPLE_RATE
HARU_DIR = Path("/content/Haru") if Path("/content").is_dir() else Path("Haru").resolve()
print("DEVICE", DEVICE, "| DTYPE", DTYPE)
print(f"frame: {FRAME_SAMPLES} samples @ {SAMPLE_RATE} Hz = {FRAME_MS:.1f} ms ({1000 / FRAME_MS:.1f} Hz)")
try:
    bf16_emul = torch.cuda.is_bf16_supported(including_emulation=True)
except TypeError:
    bf16_emul = "n/a"
print("bf16 native:", NATIVE_BF16, "| is_bf16_supported(including_emulation=True):", bf16_emul)
if not NATIVE_BF16:
    print("No native bfloat16; loading float16 weights. Linear-attn / norms / logits / MeanFlow run in fp32 after load.")
    print("Skipping emulated bf16 as the default: T4 Tensor Cores are fp16; emulation is slower at the same 2-byte storage.")


## 3. Load Haru-4B (T4-safe: one fp16 copy)

The card API is still `HaruModel.from_pretrained("higurashi-ai/Haru-4B")`. On a 15 GB T4 we **must not** call it with `device="cuda"`.

Haru's loader does this:

1. `HaruModel(config)` under `no_init_weights()` (empty tensors, default **float32**).
2. `safetensors.load_file(..., device="cuda")` — the whole **bfloat16** shard (~10.5 GB) on the GPU.
3. `load_state_dict(..., assign=True)` — model and that dict share the bf16 tensors.
4. `model.to(device="cuda", dtype=float16)` — a second copy while the bf16 shard is still alive.

That is what OOM'd a T4 (~14.4 GB allocated, 28 MiB free). 5.27B × 2 bytes is ~**9.8 GB** of float16 weights, which does fit.

This cell: download the shard, build the module on the **meta** device with `torch.set_default_dtype(DTYPE)`, copy each tensor to CUDA **already as float16**, `assign=True`, drop the dict, `empty_cache()`. The shard matches `state_dict()` (parameters plus persistent `latent_scale` / `latent_bias`). It does **not** store non-persistent buffers: Qwen3.5 RoPE `lm.rotary_emb.inv_freq` / `original_inv_freq` and the four MeanFlow `ScalarEmbed.freqs`. Those stay on meta after assign, so `model.to()` raises `Cannot copy out of meta tensor`. We recompute them on CUDA (tiny float32 tables) and share any leftover tied weights. Do not call `.to()` on the module while anything is still meta. Expected allocated after load: **~10–11.5 GB** (9.8 GB weights + CUDA context), not ~14 GB. On float16 (T4) the same cell then wraps linear-attention, RMSNorm, MLP, chunked logits, and the MeanFlow head so those ops run in fp32 without duplicating the 9.8 GB of weights (a temporary ~100 MB fp32 copy of the largest Linear). First run downloads `model.safetensors` (~10.5 GB).


In [ ]:
import gc
import math
import time

import torch.nn.functional as F
from haru.config import HaruConfig
from haru.models.haru import HaruModel
from haru.models.loaders import resolve_checkpoint
from safetensors import safe_open
from tokenizers import Tokenizer
from torch import nn
from transformers.initialization import no_init_weights


def _is_meta(t):
    return t is not None and getattr(t, "device", None) is not None and t.device.type == "meta"


def _named_meta(module):
    found = []
    for name, param in module.named_parameters():
        if _is_meta(param):
            found.append(("param", name))
    for name, buf in module.named_buffers():
        if _is_meta(buf):
            found.append(("buffer", name))
    return found


def _set_nonpersistent_buffer(module, name, tensor):
    if hasattr(torch.nn, "Buffer"):
        setattr(module, name, torch.nn.Buffer(tensor, persistent=False))
    else:
        module.register_buffer(name, tensor, persistent=False)


def _share_tied_meta_params(module):
    """assign=True can replace one tied name and leave its twin on meta."""
    loaded = {
        name: param
        for name, param in module.named_parameters()
        if param is not None and not _is_meta(param)
    }
    for name, param in list(module.named_parameters()):
        if not _is_meta(param):
            continue
        candidates = [
            tensor
            for other, tensor in loaded.items()
            if tuple(tensor.shape) == tuple(param.shape)
        ]
        embed = [tensor for other, tensor in loaded.items() if "embed_tokens" in other and tuple(tensor.shape) == tuple(param.shape)]
        if embed:
            twin = embed[0]
        elif len(candidates) == 1:
            twin = candidates[0]
        else:
            continue
        parent, _, attr = name.rpartition(".")
        owner = module.get_submodule(parent) if parent else module
        owner._parameters[attr] = twin


def _recompute_nonpersistent_buffers(module, device):
    """Fill RoPE inv_freq and MeanFlow freqs (persistent=False, omitted from the shard)."""
    device = torch.device(device)
    for child in module.modules():
        if hasattr(child, "compute_default_rope_parameters") and hasattr(child, "config"):
            inv = getattr(child, "inv_freq", None)
            orig = getattr(child, "original_inv_freq", None)
            if _is_meta(inv) or _is_meta(orig):
                inv_freq, attn = child.compute_default_rope_parameters(child.config, device=device)
                _set_nonpersistent_buffer(child, "inv_freq", inv_freq)
                _set_nonpersistent_buffer(child, "original_inv_freq", inv_freq.clone())
                child.attention_scaling = attn
        freqs = child._buffers.get("freqs") if hasattr(child, "_buffers") else None
        if _is_meta(freqs):
            half = int(freqs.shape[0])
            real = torch.exp(
                -math.log(10000.0) * torch.arange(half, dtype=torch.float32, device=device) / half
            )
            _set_nonpersistent_buffer(child, "freqs", real)


def materialize_leftover_meta(module, device):
    """Real storage for tensors the checkpoint does not contain. Never Module.to() on meta."""
    _share_tied_meta_params(module)
    for obj in (module, getattr(module, "lm", None)):
        if obj is not None and hasattr(obj, "tie_weights"):
            try:
                obj.tie_weights()
            except (NotImplementedError, RuntimeError):
                pass
    _recompute_nonpersistent_buffers(module, device)
    leftover = _named_meta(module)
    if leftover:
        raise RuntimeError(
            "still on meta after load (not in checkpoint, not recomputed): "
            + ", ".join(f"{kind}:{name}" for kind, name in leftover)
        )


def load_haru_t4(model_id, device, dtype):
    """Load Haru in `dtype` with a single GPU copy.

    Do not use HaruModel.from_pretrained(..., device="cuda", dtype=fp16) on a T4:
    haru.models.loaders.load_model does load_file(..., device=cuda) of the bf16
    shard and then model.to(dtype=fp16) while that dict is still alive.
    """
    directory = resolve_checkpoint(model_id)
    config = HaruConfig.from_file(directory / "config.json")

    prev_dtype = torch.get_default_dtype()
    torch.set_default_dtype(dtype)
    try:
        with torch.device("meta"):
            with no_init_weights():
                model = HaruModel(config)
    finally:
        torch.set_default_dtype(prev_dtype)

    weight_path = directory / "model.safetensors"
    state = {}
    with safe_open(str(weight_path), framework="pt", device="cpu") as reader:
        keys = list(reader.keys())
        for i, key in enumerate(keys, start=1):
            tensor = reader.get_tensor(key)
            if tensor.is_floating_point():
                tensor = tensor.to(dtype=dtype)
            state[key] = tensor.to(device, non_blocking=False)
            del tensor
            if i == 1 or i % 50 == 0 or i == len(keys):
                alloc = torch.cuda.memory_allocated() / 1024**3
                print(f"  tensors {i}/{len(keys)}  allocated {alloc:.2f} GB")

    model.load_state_dict(state, strict=True, assign=True)
    del state
    gc.collect()
    torch.cuda.empty_cache()

    leftover = _named_meta(model)
    print(
        f"meta leftover after assign ({len(leftover)}; not in the shard):",
        [name for _, name in leftover],
    )
    # Recompute non-persistent buffers on CUDA. Do not model.to() — leftover meta raises.
    materialize_leftover_meta(model, device)
    model.tokenizer = Tokenizer.from_file(str(directory / "tokenizer.json"))
    return model.eval().requires_grad_(False)

def _autocast_off(tensor):
    device_type = "cuda" if getattr(tensor, "is_cuda", False) else "cpu"
    return torch.amp.autocast(device_type=device_type, enabled=False)


def _fp32_causal_conv1d_fn(hidden_states, weight, bias=None, activation=None, **kwargs):
    from transformers.activations import ACT2FN

    _, hidden_size, seq_len = hidden_states.shape
    padding = weight.shape[-1] - 1
    x = hidden_states.float()
    w = weight.float().unsqueeze(1)
    b = None if bias is None else bias.float()
    out = F.conv1d(x, w, b, padding=padding, groups=hidden_size)[:, :, :seq_len]
    if activation is not None:
        out = ACT2FN[activation](out)
    return out


def _fp32_causal_conv1d_update(hidden_states, conv_state, weight, bias=None, activation=None):
    from transformers.activations import ACT2FN

    _, hidden_size, seq_len = hidden_states.shape
    state_len = conv_state.shape[-1]
    hidden_states_new = torch.cat([conv_state.float(), hidden_states.float()], dim=-1)
    conv_state.copy_(hidden_states_new[:, :, -state_len:].to(dtype=conv_state.dtype))
    w = weight.float().unsqueeze(1)
    b = None if bias is None else bias.float()
    out = F.conv1d(hidden_states_new, w, b, padding=0, groups=hidden_size)[:, :, -seq_len:]
    if activation is not None:
        out = ACT2FN[activation](out)
    return out


def _fp32_delta_rule(orig):
    def wrapped(query, key, value, g=None, beta=None, *args, **kwargs):
        state0 = kwargs.get("initial_state")
        if state0 is not None:
            kwargs = dict(kwargs)
            kwargs["initial_state"] = state0.float()
        out, state = orig(
            query.float(), key.float(), value.float(), g.float(), beta.float(), *args, **kwargs
        )
        return (None if out is None else out.float(), None if state is None else state.float())

    return wrapped


def patch_qwen35_fp32_fallbacks():
    """Keep the torch linear-attention fallback in fp32 (T4 has no FLA / causal_conv1d)."""
    import transformers.models.qwen3_5.modeling_qwen3_5 as qwen35

    if getattr(qwen35, "_t4_fp32_fallback", False):
        return
    qwen35.causal_conv1d_fn = _fp32_causal_conv1d_fn
    qwen35.causal_conv1d_update = _fp32_causal_conv1d_update
    qwen35.torch_chunk_gated_delta_rule = _fp32_delta_rule(qwen35.torch_chunk_gated_delta_rule)
    qwen35.torch_recurrent_gated_delta_rule = _fp32_delta_rule(qwen35.torch_recurrent_gated_delta_rule)
    qwen35._t4_fp32_fallback = True


def _patch_linear_fp32(mod):
    if getattr(mod, "_t4_fp32_compute", False):
        return

    def forward(x, *args, **kwargs):
        with _autocast_off(x):
            bias = None if mod.bias is None else mod.bias.float()
            return F.linear(x.float(), mod.weight.float(), bias)

    mod.forward = forward
    mod._t4_fp32_compute = True


def _chunked_logits(hidden, weight, chunk=8192):
    """fp32 logits without a 2.5 GB fp32 copy of the embedding table."""
    hidden = hidden.float()
    vocab = weight.shape[0]
    parts = []
    for i in range(0, vocab, chunk):
        parts.append(hidden @ weight[i : i + chunk].float().T)
    return torch.cat(parts, dim=-1)


def sample_token_t4(logits, temperature=0.7, top_k=25, top_p=1.0, generator=None):
    """fp32 softmax + NaN/Inf guard. Last-resort safety net around multinomial."""
    logits = torch.nan_to_num(logits.float(), nan=0.0, posinf=1e4, neginf=-1e4)
    if temperature <= 0:
        return logits.argmax(dim=-1)
    scaled = logits / temperature
    if 0 < top_k < scaled.shape[-1]:
        values, indices = torch.topk(scaled, top_k, dim=-1)
    else:
        values, indices = torch.sort(scaled, dim=-1, descending=True)
    probs = torch.softmax(values.float(), dim=-1)
    probs = torch.nan_to_num(probs, nan=0.0).clamp_min(0)
    if 0 < top_p < 1:
        keep = probs.cumsum(dim=-1) - probs < top_p
        probs = probs * keep
    denom = probs.sum(dim=-1, keepdim=True)
    safe = torch.where(denom > 0, probs / denom.clamp_min(1e-12), torch.zeros_like(probs))
    bad = denom.squeeze(-1) <= 0
    if bool(bad.any()):
        greedy = values.argmax(dim=-1, keepdim=True)
        safe = safe.clone()
        safe[bad] = 0
        safe.scatter_(1, greedy, 1.0)
    choice = torch.multinomial(safe, 1, generator=generator)
    return indices.gather(-1, choice)[:, 0]


def stabilize_haru_fp16(model):
    """Keep stored weights in fp16; run overflow-prone compute in fp32.

    T4 Tensor Cores are fp16; bf16 is emulated and slower at the same 2-byte
    storage, so we do not reload as bf16. Peak extra VRAM is a temporary fp32
    copy of the largest Linear (~100 MB) plus chunked logits (~80 MB) — well
    under ~13 GB after a ~9.9 GB load. Codec stays fp16.
    """
    if getattr(model, "_t4_fp16_stable", False):
        return model

    from transformers.activations import ACT2FN
    from transformers.models.qwen3_5.modeling_qwen3_5 import (
        Qwen3_5GatedDeltaNet,
        Qwen3_5RMSNorm,
        Qwen3_5RMSNormGated,
    )
    import haru.sampling as haru_sampling
    import haru.session as haru_session

    patch_qwen35_fp32_fallbacks()

    for name, mod in model.named_modules():
        if name == "codec" or name.startswith("codec."):
            continue
        if isinstance(mod, nn.Linear):
            _patch_linear_fp32(mod)

    for mod in model.modules():
        if isinstance(mod, Qwen3_5RMSNorm) and not getattr(mod, "_t4_fp32_norm", False):
            def make_norm(m):
                def fwd(x):
                    return m._norm(x.float()) * (1.0 + m.weight.float())

                return fwd

            mod.forward = make_norm(mod)
            mod._t4_fp32_norm = True
        if isinstance(mod, Qwen3_5RMSNormGated) and not getattr(mod, "_t4_fp32_norm", False):
            def make_gnorm(m):
                def fwd(hidden_states, gate):
                    hidden_states = hidden_states.float()
                    variance = hidden_states.pow(2).mean(-1, keepdim=True)
                    hidden_states = hidden_states * torch.rsqrt(variance + m.variance_epsilon)
                    hidden_states = m.weight.float() * hidden_states
                    hidden_states = hidden_states * ACT2FN[m.activation](gate.float())
                    return hidden_states

                return fwd

            mod.forward = make_gnorm(mod)
            mod._t4_fp32_norm = True
        if isinstance(mod, Qwen3_5GatedDeltaNet) and not getattr(mod, "_t4_fp32_gdn", False):
            orig = mod.forward

            def make_fwd(orig_fwd):
                def fwd(hidden_states, *args, **kwargs):
                    with _autocast_off(hidden_states):
                        return orig_fwd(hidden_states.float(), *args, **kwargs)

                return fwd

            mod.forward = make_fwd(orig)
            mod._t4_fp32_gdn = True

    orig_embed = model.embed
    orig_forward = model.forward
    orig_head_sample = model.head.sample
    weight = model.lm.embed_tokens.weight

    def embed(*args, **kwargs):
        return orig_embed(*args, **kwargs).float()

    def forward(embeddings, cache):
        return orig_forward(embeddings.float(), cache)

    def logits(hidden):
        with _autocast_off(hidden):
            return _chunked_logits(hidden, weight)

    def head_sample(hidden, noise, guidance=None, steps=None):
        with _autocast_off(hidden):
            return orig_head_sample(hidden.float(), noise.float(), guidance, steps)

    model.embed = embed
    model.forward = forward
    model.logits = logits
    model.head.sample = head_sample

    if not getattr(haru_sampling, "_t4_safe_sample", False):
        haru_sampling.sample_token = sample_token_t4
        haru_session.sample_token = sample_token_t4
        haru_sampling._t4_safe_sample = True

    model._t4_fp16_stable = True
    n_lin = sum(1 for m in model.modules() if getattr(m, "_t4_fp32_compute", False))
    print(
        f"T4 fp16 stability: {n_lin} Linear modules compute in fp32; "
        "gated-delta-net / norms / logits / MeanFlow stay fp32; weights remain float16. "
        "sample_token has an fp32 softmax + NaN guard (safety net, not the main fix)."
    )
    return model


if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_vram = lambda label: print(
    f"VRAM {label}: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB | "
    f"free {torch.cuda.mem_get_info()[0] / 1024**3:.1f}/{torch.cuda.mem_get_info()[1] / 1024**3:.1f} GB"
)
print_vram("before load")
print(f"loading {MODEL_ID} as {DTYPE} onto {DEVICE} (meta construct, one GPU copy)")
print("expected after load: ~10–11.5 GB allocated (9.8 GB fp16 weights + CUDA context)")

t0 = time.perf_counter()
model = load_haru_t4(MODEL_ID, DEVICE, DTYPE)
if DTYPE == torch.float16:
    stabilize_haru_fp16(model)
torch.cuda.synchronize()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.1f}s")

n_params = sum(p.numel() for p in model.parameters())
bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print("class:", type(model).__name__)
print(f"parameters: {n_params:,} ({n_params / 1e9:.3f}B; Hub 5.270B incl. codec)")
print(f"parameter storage: {bytes / 1024**3:.2f} GB at {model.dtype}")
print("sample_rate:", model.config.sample_rate, "| frame_samples:", model.config.frame_samples)
print("dtype:", model.dtype, "| device:", model.device)
print("lm layers:", model.config.lm.get("num_hidden_layers"), "| hidden:", model.config.hidden_size)
print_vram("after load")
print("Card figure is ~12 GB on an NVIDIA GPU; the number above is this session so far.")


## 3.1 Load check

Assert no parameter or buffer is still on the meta device, then print `torch.cuda.memory_allocated` and `max_memory_allocated`.


In [ ]:
meta = _named_meta(model)
assert not meta, f"meta tensors remain: {meta}"
print("no meta tensors remain")
print(f"torch.cuda.memory_allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"torch.cuda.max_memory_allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")


## 4. Reference voice, user clip, and character cards

Haru ships **without** voice clips. The card: one speaker, 3 seconds to 3 minutes, clean speech; only use a voice you have the right to use.

**Source.** Both the voice reference and the user utterance come from [`hf-internal-testing/librispeech_asr_dummy`](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy) (`clean` / `validation`, 73 clips, ~9 MB). These are public-domain LibriVox audiobook readings, the same family as OpenSLR LibriSpeech (CC BY 4.0). This is **not** the official `test-clean` leaderboard split. That dummy split is **one speaker** (`1272`); the cell uses two different clips as voice A and voice B, and concatenates short clips if needed so each reference is at least 3 s. Clips are 16 kHz on disk; `haru.audio.load_audio` resamples them to 24 kHz.

The **system prompt** is a character card with a fixed layout. The [repository README](https://github.com/higurashi-ai/Haru/blob/main/README.md#the-system-prompt) builds one with `render_card`. Demo 1 uses that README example. Demo 2 uses the bundled English `chat` card from the Haru package (`haru/assets/cards/en.json`).


In [ ]:
import io
import json
from collections import defaultdict
from importlib import resources
from pathlib import Path

import numpy as np
import soundfile as sf
from datasets import Audio as HfAudio, load_dataset
from haru.audio import load_audio, prepare_reference, save_audio
from haru.prompt import card_templates, render_card
from IPython.display import Audio, display

SAMPLE_DIR = Path("haru_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
MIN_REF_S = 3.0

ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation",
)
# datasets 4/5 decode via torchcodec; keep encoded bytes and decode with soundfile.
ds = ds.cast_column("audio", HfAudio(decode=False))


def _row_get(row, key, default=None):
    if isinstance(row, dict):
        return row.get(key, default)
    if key in row:
        return row[key]
    return default


def speaker_id_of(row):
    """speaker_id column, else the LibriSpeech speaker prefix of id/file."""
    sid = _row_get(row, "speaker_id")
    if sid is not None and str(sid).strip() not in ("", "None"):
        try:
            return str(int(sid))
        except (TypeError, ValueError):
            text = str(sid).strip()
            if text:
                return text
    for key in ("id", "file"):
        val = _row_get(row, key)
        if not val:
            continue
        prefix = Path(str(val)).stem.split("-")[0]
        if prefix.isdigit():
            return prefix
    return "unknown"


def _as_mono(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        return arr.reshape(-1)
    time_axis = int(np.argmax(arr.shape))
    axes = tuple(i for i in range(arr.ndim) if i != time_axis)
    return np.asarray(arr.mean(axis=axes), dtype=np.float32).reshape(-1)


def decode_pcm(audio):
    """Mono float32 PCM and native sampling rate from a datasets Audio value."""
    if audio is None:
        raise ValueError("missing audio")
    if isinstance(audio, dict):
        if audio.get("array") is not None:
            sr = int(audio.get("sampling_rate") or audio.get("sample_rate") or 16000)
            return _as_mono(audio["array"]), sr
        data = audio.get("bytes")
        path = audio.get("path")
        if not data and path and Path(path).is_file():
            data = Path(path).read_bytes()
        if data:
            arr, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)
            return _as_mono(arr), int(sr)
    arr = getattr(audio, "array", None)
    sr = getattr(audio, "sampling_rate", None) or getattr(audio, "sample_rate", None)
    if arr is not None and sr:
        return _as_mono(arr), int(sr)
    raise ValueError(f"could not decode audio: {type(audio)!r}")


def concat_to_min_seconds(recs, min_s=MIN_REF_S):
    if not recs:
        raise ValueError("no clips to concatenate")
    sr0 = recs[0]["sr"]
    chunks, used, total = [], [], 0.0
    for rec in recs:
        pcm, sr = rec["pcm"], rec["sr"]
        if sr != sr0:
            from math import gcd
            from scipy.signal import resample_poly

            g = gcd(int(sr), int(sr0))
            pcm = resample_poly(pcm, int(sr0) // g, int(sr) // g).astype(np.float32)
        chunks.append(pcm)
        used.append(rec)
        total += pcm.shape[0] / float(sr0)
        if total >= min_s:
            break
    return np.concatenate(chunks), sr0, used


def reference_from(recs):
    recs = sorted(recs, key=lambda r: -r["seconds"])
    if recs[0]["seconds"] >= MIN_REF_S:
        r = recs[0]
        return r["pcm"], r["sr"], [r]
    return concat_to_min_seconds(recs, MIN_REF_S)


records = []
for i, row in enumerate(ds):
    pcm, sr = decode_pcm(row["audio"])
    records.append(
        {
            "id": str(_row_get(row, "id") or _row_get(row, "file") or i),
            "speaker_id": speaker_id_of(row),
            "text": _row_get(row, "text", "") or "",
            "pcm": pcm,
            "sr": sr,
            "seconds": pcm.shape[0] / float(sr),
        }
    )

by_speaker = defaultdict(list)
for rec in records:
    by_speaker[rec["speaker_id"]].append(rec)

eligible = [
    sid
    for sid, recs in by_speaker.items()
    if sum(r["seconds"] for r in recs) >= MIN_REF_S or any(r["seconds"] >= MIN_REF_S for r in recs)
]
eligible.sort(key=lambda sid: -max(r["seconds"] for r in by_speaker[sid]))

if len(eligible) >= 2:
    voice_a_pcm, voice_a_sr, voice_a_used = reference_from(by_speaker[eligible[0]])
    voice_b_pcm, voice_b_sr, voice_b_used = reference_from(by_speaker[eligible[1]])
    print(
        f"Voices from speakers {eligible[0]} and {eligible[1]} "
        f"({len(by_speaker)} speaker id(s) in the dummy split)."
    )
else:
    # This dummy split is a single speaker (1272). Use two different clips.
    ranked = sorted(records, key=lambda r: -r["seconds"])
    voice_a_pcm, voice_a_sr, voice_a_used = reference_from(ranked)
    used_ids = {r["id"] for r in voice_a_used}
    rest = [r for r in ranked if r["id"] not in used_ids] or ranked[1:] or ranked
    voice_b_pcm, voice_b_sr, voice_b_used = reference_from(rest)
    print(
        f"Dummy split has {len(by_speaker)} speaker id(s) {sorted(by_speaker)}; "
        "using two different clips as voice A and voice B."
    )

used_ids = {r["id"] for r in voice_a_used + voice_b_used}
user_pool = [r for r in records if r["id"] not in used_ids]
if not user_pool:
    user_pool = [r for r in records if r["id"] != voice_a_used[0]["id"]] or records
user_rec = sorted(user_pool, key=lambda r: r["seconds"])[len(user_pool) // 2]


def write_pcm(path, pcm, sr):
    pcm = np.asarray(pcm, dtype=np.float32).reshape(-1)
    sf.write(path, pcm, int(sr))
    return path


raw_voice_a = write_pcm(SAMPLE_DIR / "voice-a-librispeech.wav", voice_a_pcm, voice_a_sr)
raw_voice_b = write_pcm(SAMPLE_DIR / "voice-b-librispeech.wav", voice_b_pcm, voice_b_sr)
raw_user = write_pcm(SAMPLE_DIR / "user-librispeech.wav", user_rec["pcm"], user_rec["sr"])

voice_a = prepare_reference(load_audio(raw_voice_a))
voice_b = prepare_reference(load_audio(raw_voice_b))
user_audio = load_audio(raw_user)
save_audio(SAMPLE_DIR / "voice-a-24k.wav", voice_a)
save_audio(SAMPLE_DIR / "voice-b-24k.wav", voice_b)
save_audio(SAMPLE_DIR / "user-24k.wav", user_audio)


def describe_voice(label, used, pcm):
    ids = ", ".join(r["id"] for r in used)
    dummy_s = sum(r["seconds"] for r in used)
    print(
        f"{label}: id={ids} speaker={used[0]['speaker_id']} "
        f"| dummy {dummy_s:.2f}s @ {used[0]['sr']} Hz "
        f"| 24 kHz {len(pcm) / SAMPLE_RATE:.2f}s"
    )
    for rec in used:
        print(f"  transcript: {rec['text']}")


print("Source: hf-internal-testing/librispeech_asr_dummy (LibriVox / public domain, CC BY 4.0 family).")
describe_voice("voice A (demo 1)", voice_a_used, voice_a)
describe_voice("voice B (demo 2)", voice_b_used, voice_b)
describe_voice("user utterance", [user_rec], user_audio)
print()
print("User clip (what the model will hear):")
display(Audio(str(SAMPLE_DIR / "user-24k.wav")))
print("Voice A reference:")
display(Audio(str(SAMPLE_DIR / "voice-a-24k.wav")))

templates = card_templates()["templates"]
card_readme = render_card(
    {
        "persona": "Haru is a 29-year-old nurse in Denver. She is warm, chatty and laughs easily.",
        "name": "Haru",
        "gender": "female",
        "other_name": "Sam",
        "relationship": "Old friends from high school.",
        "situation": "A phone call to catch up after a long time.",
    },
    templates["en"],
)
bundled = json.loads((resources.files("haru") / "assets" / "cards" / "en.json").read_text(encoding="utf-8"))
card_bundled = render_card(bundled["chat"], templates["en"])
print()
print("--- demo 1 card (README example) ---")
print(card_readme)
print()
print("--- demo 2 card (bundled en.json chat) ---")
print(card_bundled)


## 5. Offline duplex helper

The card loop is:

```python
session = DuplexSession(model, system_prompt=card, reference=voice)
frame = session.start()
while not frame.done:
    frame = session.step(user_audio)  # 3200 samples at 24 kHz in, 3200 out
```

The model is **one frame ahead** of the user: `start()` returns frame 0, and each `step()` with user frame `k` returns model frame `k + 1`. The official `haru-inference` CLI therefore runs `start()` plus `N - 1` steps for an `N`-frame recording so the output is the same length as the input. We follow that, after padding the utterance with trailing silence so the model has time to reply.

Colab cannot do live mic duplex. This is the same Python API, fed from a file.


In [ ]:
import numpy as np
from haru import DuplexConfig, DuplexSession


def pad_user_frames(pcm, frame_samples=FRAME_SAMPLES, trailing_silence_s=4.0):
    pcm = np.asarray(pcm, dtype=np.float32).reshape(-1)
    silence = np.zeros(int(round(trailing_silence_s * SAMPLE_RATE)), dtype=np.float32)
    pcm = np.concatenate([pcm, silence])
    n = max(1, -(-len(pcm) // frame_samples))
    pcm = np.pad(pcm, (0, n * frame_samples - len(pcm)))
    return pcm.reshape(n, frame_samples)


def run_offline_session(model, card, voice, user_pcm, *, trailing_silence_s=4.0, seed=0, label=""):
    chunks = pad_user_frames(user_pcm, model.config.frame_samples, trailing_silence_s)
    n = int(chunks.shape[0])
    config = DuplexConfig(
        max_frames=n,
        seed=seed,
        codec_backend="torch",
        guidance_scale=1.5,
    )
    session = DuplexSession(model, system_prompt=card, reference=voice, config=config)
    print(f"{label}frames in (utterance + {trailing_silence_s:.1f}s silence): {n} ({n * FRAME_MS / 1000:.2f}s)")
    torch.cuda.synchronize()
    frames = [session.start()]
    for i in range(n - 1):
        frames.append(session.step(chunks[i]))
        if (i + 1) % 15 == 0 or i + 1 == n - 1:
            text = "".join(f.text for f in frames)
            print(f"  step {i + 1}/{n - 1}  last {frames[-1].seconds * 1000:.0f} ms  text so far: {text!r}")
    torch.cuda.synchronize()
    reply = np.concatenate([f.audio for f in frames])
    text = "".join(f.text for f in frames)
    step_s = [f.seconds for f in frames[1:]]
    return frames, reply, text, step_s, chunks


def show_text_stream(frames):
    print("per-frame text (empty = padding / not speaking):")
    for f in frames:
        if f.text:
            print(f"  [{f.index:4d} | {f.index * FRAME_MS:7.0f} ms] {f.text!r}")
    joined = "".join(f.text for f in frames).strip()
    print()
    print("joined text:", joined if joined else "(no text tokens in this run)")
    return joined


## 5.1 Finite-logits sanity

One forward step plus a short `session.start()` before Demo 1. Asserts text logits are finite. If they are not, prints max |activation| per decoder block so an overflow is diagnosable. Demo 1 still uses the full voice clip; this cell uses a 3 s slice so it stays quick.


In [ ]:
import haru.session as haru_session


def _layer_hooks(model):
    stats = []
    hooks = []

    def make(i):
        def _h(mod, inp, out):
            t = out[0] if isinstance(out, tuple) else out
            if not torch.is_tensor(t):
                return
            x = t.detach().float()
            while len(stats) <= i:
                stats.append((0.0, True, ()))
            mx = float(x.abs().max().cpu()) if x.numel() else 0.0
            stats[i] = (mx, bool(torch.isfinite(x).all().cpu()), tuple(x.shape))

        return _h

    for i, layer in enumerate(model.lm.layers):
        hooks.append(layer.register_forward_hook(make(i)))
    return stats, hooks


def _print_layer_maxabs(stats):
    print("max |activation| per decoder layer:")
    for i, (mx, finite, shape) in enumerate(stats):
        flag = "" if finite else "  NONFINITE"
        print(f"  layer {i:02d}  maxabs={mx:.4e}  finite={finite}  shape={shape}{flag}")


def _assert_finite(name, tensor, stats):
    ok = bool(torch.isfinite(tensor).all().cpu())
    mx = float(tensor.detach().float().abs().max().cpu()) if tensor.numel() else 0.0
    print(f"{name}: dtype={tensor.dtype} shape={tuple(tensor.shape)} finite={ok} maxabs={mx:.4e}")
    if not ok:
        _print_layer_maxabs(stats)
    assert ok, f"{name} is not finite (fp16 overflow). See per-layer maxabs above."
    return mx


stats, hooks = _layer_hooks(model)
try:
    pad = torch.tensor([[model.config.pad_token_id]], device=model.device)
    hidden = model(model.embed(pad), model.new_cache())
    logits = model.logits(hidden)
    print("one-token forward")
    _assert_finite("hidden", hidden, stats)
    _assert_finite("logits", logits, stats)
finally:
    for h in hooks:
        h.remove()

# start() is the Demo 1 crash site (prefill then sample_token / multinomial)
voice_probe = voice_a[: min(len(voice_a), 3 * SAMPLE_RATE)]
print(f"short start(): {len(voice_probe) / SAMPLE_RATE:.2f}s reference, max_frames=2")
stats, hooks = _layer_hooks(model)
captured = {}
_orig_sample = haru_session.sample_token


def _capture_sample(logits, *args, **kwargs):
    captured["logits"] = logits.detach()
    return _orig_sample(logits, *args, **kwargs)


haru_session.sample_token = _capture_sample
try:
    cfg = DuplexConfig(max_frames=2, seed=0, codec_backend="torch", guidance_scale=1.5)
    sess = DuplexSession(model, system_prompt=card_readme, reference=voice_probe, config=cfg)
    frame0 = sess.start()
    if "logits" in captured:
        _assert_finite("start() logits", captured["logits"], stats)
    audio = torch.as_tensor(frame0.audio)
    print(
        f"frame 0: token={frame0.token} audio_finite={bool(torch.isfinite(audio).all())} "
        f"text={frame0.text!r} {frame0.seconds * 1000:.0f} ms"
    )
    if not torch.isfinite(audio).all():
        _print_layer_maxabs(stats)
        raise AssertionError("start() audio is not finite")
except Exception:
    if stats:
        _print_layer_maxabs(stats)
    raise
finally:
    haru_session.sample_token = _orig_sample
    for h in hooks:
        h.remove()

print("sanity passed: logits are finite")
if torch.cuda.is_available():
    print(
        f"VRAM after sanity: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"peak {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )


## 6. Demo 1 — English utterance, README character card

Feed the LibriSpeech user clip frame by frame (3200 samples at 24 kHz), then **4 s of silence** so the model can reply. Character card is the [README example](https://github.com/higurashi-ai/Haru/blob/main/README.md#the-system-prompt) (nurse in Denver). Voice reference is dummy speaker A, resampled to 24 kHz.

This is a **research base model**. It may backchannel, talk over the user, or wander; it is not an instruction-following assistant.


In [ ]:
print("Feeding user-24k.wav with the README nurse card and voice A.")
print("Trailing silence: 4.0 s. Seed: 0.")
print()

frames_1, reply_1, text_1, steps_1, chunks_1 = run_offline_session(
    model,
    card_readme,
    voice_a,
    user_audio,
    trailing_silence_s=4.0,
    seed=0,
    label="demo 1: ",
)

out_1 = SAMPLE_DIR / "demo1-model.wav"
save_audio(out_1, reply_1)
print()
print(f"wrote {out_1} ({len(reply_1) / SAMPLE_RATE:.2f}s, {SAMPLE_RATE} Hz)")
if reply_1.size == 0 or not np.isfinite(reply_1).all():
    print("WARNING: demo 1 audio is empty or non-finite. Re-run the load cell (fp16 weights + fp32 linear-attn / logits). The sanity cell before Demo 1 prints per-layer maxabs.")
show_text_stream(frames_1)
print()
print("model audio:")
display(Audio(str(out_1)))
print(
    f"VRAM allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 7. Demo 2 — same user clip, different card and voice

Same user utterance as demo 1. Character card is the bundled English `chat` card shipped in the Haru package (college-student first meeting). Voice reference is the second dummy clip (voice B). That is the card's two controls: **text card** for character, **reference clip** for voice.


In [ ]:
print("Same user clip, bundled en.json chat card, voice B.")
print("Trailing silence: 4.0 s. Seed: 1.")
print()

frames_2, reply_2, text_2, steps_2, chunks_2 = run_offline_session(
    model,
    card_bundled,
    voice_b,
    user_audio,
    trailing_silence_s=4.0,
    seed=1,
    label="demo 2: ",
)

out_2 = SAMPLE_DIR / "demo2-model.wav"
save_audio(out_2, reply_2)
print()
print(f"wrote {out_2} ({len(reply_2) / SAMPLE_RATE:.2f}s, {SAMPLE_RATE} Hz)")
if reply_2.size == 0 or not np.isfinite(reply_2).all():
    print("WARNING: demo 2 audio is empty or non-finite. Re-run the load cell (fp16 weights + fp32 linear-attn / logits). The sanity cell before Demo 1 prints per-layer maxabs.")
show_text_stream(frames_2)
print()
print("voice B reference:")
display(Audio(str(SAMPLE_DIR / "voice-b-24k.wav")))
print("model audio:")
display(Audio(str(out_2)))
print(
    f"VRAM allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 8. Demo 3 — per-frame `step` time vs the 133 ms budget

Full-duplex at 7.5 Hz means each `session.step` has a **133.3 ms** realtime budget (`3200 / 24000` s). `Frame.seconds` is that step's compute time. `start()` is prompt prefill (character card + voice reference) and is **not** in the per-frame budget.

These numbers are **this T4 session**, not the card. The technical blog's ~65 ms end-to-end figure is an M1 Max native app with int8 weights; the ~15 ms GPU figure is an NVIDIA B300 with vLLM-Omni. Neither is a free Colab T4 measurement.


In [ ]:
def report_steps(label, frames, step_s):
    start_s = frames[0].seconds
    arr = np.asarray(step_s, dtype=np.float64)
    if arr.size == 0:
        print(f"{label}: no step() calls")
        return
    under = int((arr * 1000.0 <= FRAME_MS).sum())
    print(f"{label}")
    print(f"  start() prefill: {start_s * 1000:.0f} ms  (not the 133 ms budget)")
    print(f"  step() n={arr.size}  mean {arr.mean() * 1000:.0f} ms  "
          f"median {np.median(arr) * 1000:.0f} ms  "
          f"p90 {np.percentile(arr, 90) * 1000:.0f} ms  "
          f"max {arr.max() * 1000:.0f} ms")
    print(f"  realtime budget: {FRAME_MS:.1f} ms/frame  |  "
          f"{under}/{arr.size} steps at or under budget")
    realtime = bool(np.median(arr) * 1000.0 <= FRAME_MS)
    print(f"  median vs budget: {'under' if realtime else 'over'} (T4 measurement, not a published benchmark)")


print(f"dtype {DTYPE} on {torch.cuda.get_device_name(0)}  |  codec_backend=torch")
print()
report_steps("demo 1 (README card, voice A)", frames_1, steps_1)
print()
report_steps("demo 2 (bundled chat card, voice B)", frames_2, steps_2)
print()
print(
    f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB  "
    f"(card: about 12 GB on an NVIDIA GPU)"
)


## 9. Caveats and cleanup

- **Research base model.** Not an instruction-following assistant. The card: it can hold a natural conversation, but it is not a reliable source of facts. Do not use it as a nurse, a tutor, or a source of advice — the README card is a *character*, not a qualified professional.
- **Licence.** Apache-2.0 on the weights and code.
- **Install.** Card: `git clone https://github.com/higurashi-ai/Haru && pip install -e ./haru`. This notebook pins commit `7fe6b7665f8bf035998f401f8cb779fbe53f2751`. Keep Colab's torch. Use `%pip`, not `python -I`. Needs `transformers>=5.16`. Restart after install (the install cell does this with `os.kill`); do not `import haru` before that restart.
- **API.** Inference is still `DuplexSession(model, system_prompt=card, reference=voice)`. `start()` then `step(3200 samples)` until `frame.done`. Character cards go through `render_card` (fixed field order). On a T4 we do **not** call `HaruModel.from_pretrained(..., device="cuda")` (that path double-occupies VRAM).
- **T4 / dtype.** Checkpoint is bfloat16. This notebook builds on meta and loads **float16** onto the GPU in one copy (~9.8 GB weights; expect ~10–11.5 GB allocated after load). Linear-attention, RMSNorm, MLP, chunked logits, softmax, and the MeanFlow head then run in fp32 so gated delta-net does not overflow into NaN (which crashed `torch.multinomial` on a T4). Emulated bf16 is not the default: same storage, slower, no Tensor Cores. The Triton codec backend is skipped (needs bfloat16 on CUDA).
- **No live mic.** Colab cannot run the web demo's duplex loop. Offline file → frames → WAV only.
- **Voice clips.** Haru ships none. References here are `hf-internal-testing/librispeech_asr_dummy` (LibriVox / public domain). Only clone a voice you have the right to use.
- **Languages.** en, es, pt, de, fr, zh, ja, ru, ko on the card. This notebook's spoken demos are English LibriSpeech.
- **Not LFM3-Audio, not ZONOS3-Mini, not Fluid Speech Pro.** Those are ASR/TTS stacks, not native full-duplex speech-to-speech.

Free the GPU before you open another notebook. Peak VRAM below is `torch.cuda.max_memory_allocated` for this session.


In [ ]:
import gc

if torch.cuda.is_available():
    print(f"before cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

for name in ("model", "session"):
    if name in globals():
        del globals()[name]
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"after cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )
else:
    print("CPU runtime — nothing to empty on the GPU.")
